
## Trajectory Difference Analysis




In [ ]:
!pip -q install pandas numpy scipy pyproj frechetdist


In [ ]:
# Import libraries
import numpy as np
import pandas as pd

from pathlib import Path
from pyproj import Transformer
from frechetdist import frdist
from scipy.interpolate import interp1d


In [ ]:
# Select platform
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


Mounted at /content/drive


In [ ]:
# Project path
if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/Final_Thesis")
else:
    PROJECT_ROOT = Path.cwd()

print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)


Running in Colab: True
Project root: /content/drive/MyDrive/Final_Thesis


In [ ]:
# Input paths
RUNS_PATH = PROJECT_ROOT / "runs"
POSITION_PATH = RUNS_PATH / "PositionValue"

manual_path = PROJECT_ROOT / "vdo" / "clip" / "output.csv"  # change the file for linkoping video
drone_path = POSITION_PATH / "try_center_dronedata_1hz.csv" # change the file for linkoping video
gnss_path = POSITION_PATH / "gnss_61bus.csv"                # change the file for linkoping video

# Analysis settings
TRACK_ID = 10
START_DATETIME = "2026-03-12 07:10:00"

print("Manual trajectory:", manual_path)
print("Drone trajectory:", drone_path)
print("GNSS trajectory:", gnss_path)


Manual trajectory: /content/drive/MyDrive/Final_Thesis/vdo/clip/output.csv
Drone trajectory: /content/drive/MyDrive/Final_Thesis/runs/PositionValue/try_center_dronedata_1hz.csv
GNSS trajectory: /content/drive/MyDrive/Final_Thesis/runs/PositionValue/gnss_61bus.csv


In [ ]:
# Load trajectories
df_m = pd.read_csv(manual_path)
df_d = pd.read_csv(drone_path)
df_g = pd.read_csv(gnss_path)

# Select the tracked bus used in the trajectory comparison
df_d = df_d[df_d["track_id"] == TRACK_ID].copy()

print("Manual rows:", len(df_m))
print("Drone rows:", len(df_d))
print("GNSS rows:", len(df_g))


Manual rows: 289
Drone rows: 472
GNSS rows: 431


### 1. Column configuration and time alignment

The manual trajectory already uses relative time in seconds. The drone trajectory contains clock time, while the GNSS data contains a full timestamp. All three are converted to seconds relative to the selected analysis start time.


In [ ]:
# Column configuration
manual_time_col = "time_sec"
drone_time_col = "time"
gnss_time_col = "datetime_l"

m_lat_col, m_lon_col = "latitude", "longitude"
d_lat_col, d_lon_col = "latitude", "longitude"
g_lat_col, g_lon_col = "lat", "lon"

start_dt = pd.to_datetime(START_DATETIME)

# Manual: already relative seconds
df_m["time_s"] = pd.to_numeric(df_m[manual_time_col], errors="coerce")

# Drone: clock time only
drone_dt = pd.to_datetime(
    start_dt.strftime("%Y-%m-%d") + " " + df_d[drone_time_col].astype(str),
    errors="coerce"
)
df_d["time_s"] = (drone_dt - start_dt).dt.total_seconds()

# GNSS: full datetime
gnss_dt = pd.to_datetime(df_g[gnss_time_col].astype(str), errors="coerce")
df_g["time_s"] = (gnss_dt - start_dt).dt.total_seconds()


### 2. Convert latitude/longitude to metric coordinates

Coordinates are transformed from WGS84 (`EPSG:4326`) to UTM Zone 33N (`EPSG:32633`) so that trajectory differences can be measured in metres.


In [ ]:
transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32633",
    always_xy=True
)

def add_xy(df, lat_col, lon_col):
    out = df.copy()
    out["x"], out["y"] = transformer.transform(
        out[lon_col].astype(float).to_numpy(),
        out[lat_col].astype(float).to_numpy()
    )
    return out

df_m = add_xy(df_m, m_lat_col, m_lon_col)
df_d = add_xy(df_d, d_lat_col, d_lon_col)
df_g = add_xy(df_g, g_lat_col, g_lon_col)


### 3. Clean trajectories and select the common time window


In [ ]:
def clean_trajectory(df):
    out = df[["time_s", "x", "y"]].copy()
    out = out.dropna()
    out = out.sort_values("time_s")
    out = out.drop_duplicates(subset="time_s")
    return out

manual = clean_trajectory(df_m)
drone = clean_trajectory(df_d)
gnss = clean_trajectory(df_g)

print(f"Manual time range: {manual.time_s.min():.1f} to {manual.time_s.max():.1f} s")
print(f"Drone time range:  {drone.time_s.min():.1f} to {drone.time_s.max():.1f} s")
print(f"GNSS time range:   {gnss.time_s.min():.1f} to {gnss.time_s.max():.1f} s")

start_common = max(
    manual.time_s.min(),
    drone.time_s.min(),
    gnss.time_s.min()
)

end_common = min(
    manual.time_s.max(),
    drone.time_s.max(),
    gnss.time_s.max()
)

manual_trim = manual[
    (manual.time_s >= start_common) &
    (manual.time_s <= end_common)
].copy()

drone_trim = drone[
    (drone.time_s >= start_common) &
    (drone.time_s <= end_common)
].copy()

gnss_trim = gnss[
    (gnss.time_s >= start_common) &
    (gnss.time_s <= end_common)
].copy()

print(f"\nCommon time window: {start_common:.1f} to {end_common:.1f} s")
print("Points after trimming:")
print("Manual:", len(manual_trim))
print("Drone:", len(drone_trim))
print("GNSS:", len(gnss_trim))


Manual time range: 1.0 to 470.0 s
Drone time range:  0.0 to 470.0 s
GNSS time range:   -64.0 to 400.0 s

Common time window: 1.0 to 400.0 s
Points after trimming:
Manual: 237
Drone: 400
GNSS: 315


### 4. Interpolate trajectories to a common 1-second interval


In [ ]:
def interpolate_trajectory(df, common_times):
    fx = interp1d(
        df["time_s"],
        df["x"],
        bounds_error=False,
        fill_value="extrapolate"
    )
    fy = interp1d(
        df["time_s"],
        df["y"],
        bounds_error=False,
        fill_value="extrapolate"
    )

    return pd.DataFrame({
        "time_s": common_times,
        "x": fx(common_times),
        "y": fy(common_times)
    })

common_times = np.arange(
    np.ceil(start_common),
    np.floor(end_common) + 1,
    1
)

manual_i = interpolate_trajectory(manual_trim, common_times)
drone_i = interpolate_trajectory(drone_trim, common_times)
gnss_i = interpolate_trajectory(gnss_trim, common_times)

print("Matched points:", len(common_times))


Matched points: 400


### 5. Point-wise trajectory difference metrics





In [ ]:
def trajectory_error_metrics(test, reference):
    dx = test["x"].to_numpy() - reference["x"].to_numpy()
    dy = test["y"].to_numpy() - reference["y"].to_numpy()
    distance = np.sqrt(dx**2 + dy**2)

    return {
        "Mean Error (m)": np.mean(distance),
        "Median Error (m)": np.median(distance),
        "RMSE (m)": np.sqrt(np.mean(distance**2)),
        "95th Percentile Error (m)": np.percentile(distance, 95),
        "Max Error (m)": np.max(distance)
    }

difference_results = pd.DataFrame({
    "Drone/YOLO vs Manual": trajectory_error_metrics(drone_i, manual_i),
    "GNSS vs Manual": trajectory_error_metrics(gnss_i, manual_i),
    "GNSS vs Drone/YOLO": trajectory_error_metrics(gnss_i, drone_i)
})

difference_results.round(3)


,Drone/YOLO vs Manual,GNSS vs Manual,GNSS vs Drone/YOLO
Mean Error (m),3.058,13.313,12.581
Median Error (m),3.090,5.844,4.749
RMSE (m),3.087,20.857,20.782
95th Percentile Error (m),3.914,53.440,54.000
Max Error (m),4.818,87.466,90.061
